Import Required Libraries

In [41]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import os

Define API Credentials

In [42]:
CLIENT_ID = "YOUR_ID"          
CLIENT_SECRET = "YOUR_SECRECT"
REDIRECT_URI = "http://127.0.0.1:3000/callback"

Spotify OAuth Authentication|

In [43]:
from spotipy.oauth2 import SpotifyOAuth
import spotipy


sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri="http://127.0.0.1:3000/callback"
))

track_id = "3n3Ppam7vgaVa1iaRUc9Lp"  # Eminem – Without Me
features = sp.audio_features([track_id])[0]
print(features)

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3n3Ppam7vgaVa1iaRUc9Lp with Params: {} returned 403 due to None


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=3n3Ppam7vgaVa1iaRUc9Lp:
 None, reason: None

Define Audio Feature Extraction Function

In [ ]:
def get_audio_features_from_query(query: str) -> dict:
    """
    Search for a track by query string and retrieve its audio features.
    """
    result = sp.search(q=query, type='track', limit=1)
    if not result['tracks']['items']:
        return None

    track = result['tracks']['items'][0]
    track_id = track['id']
    features = sp.audio_features([track_id])[0]
    
    # Add metadata
    features['track_name'] = track['name']
    features['artist'] = track['artists'][0]['name']
    return features

Build Bass-Rich Track Candidate Pool

In [ ]:
track_list = [
    "Baby Come Back - Player",
    "Believe - Two Ton Shoe",
    "Teacher - Nick Jonas",
    "Them Changes - Thundercat",
    "Smoke on the Water - Deep Purple",
    "Loretta - Ginger Root",
    "Le Freak - CHIC",
    "Saturnine - Justice, Miguel"
]

Data Fetch Helpers

In [ ]:
def get_audio_features_from_query(query):
    try:
        result = sp.search(q=query, limit=1, type='track')
        items = result.get('tracks', {}).get('items', [])
        if not items:
            return None

        track = items[0]
        track_id = track['id']
        features = sp.audio_features([track_id])[0]

        return {
            'id': track_id,
            'track_name': track['name'],
            'artist': track['artists'][0]['name'],
            'loudness': features['loudness'],
            'energy': features['energy'],
            'danceability': features['danceability'],
            'valence': features['valence'],
            'acousticness': features['acousticness'],
            'instrumentalness': features['instrumentalness']
        }

    except Exception as e:
        print(f"⚠️ Error for query '{query}': {e}")
        return None

Collect Audio Features for Candidates

In [ ]:
feature_cols = [
    'id', 'track_name', 'artist',
    'loudness', 'energy', 'danceability',
    'valence', 'acousticness', 'instrumentalness'
]

results = []

for query in track_list:
    f = get_audio_features_from_query(query)
    if f:
        results.append({col: f.get(col, None) for col in feature_cols})
    else:
        print(f"❌ Not found: {query}")

candidate_df = pd.DataFrame(results)